In [ ]:
import pandas as pd


class Icd9Encoder(BaseEstimator, TransformerMixin):

    def __init__(self, comorbidity_df=None):
        self.comorbidity_df = comorbidity_df  # safe, sklearn cloneable

    def fit(self, X, y):
        df = X.copy().reset_index(drop = True)
        y = y.reset_index(drop = True)
        df['target'] = y.values

        com = self.comorbidity_df   # ACCESS HERE SAFELY

        merged = df[['subject_id','hadm_id','target']].merge(
            com, on=['subject_id','hadm_id'], how='left'
        )

        self.icd_mortality_ = merged.groupby('icd9_code')['target'].mean()
        self.global_mean_ = df['target'].mean()
        return self

    def transform(self, X):
        df = X.copy().reset_index(drop = True)
        com = self.comorbidity_df

        merged = df[['subject_id','hadm_id']].merge(
            com, on=['subject_id','hadm_id'], how='left'
        )

        merged['mortality_proxy'] = merged['icd9_code'].map(self.icd_mortality_)
        merged['mortality_proxy'] = merged['mortality_proxy'].fillna(self.global_mean_)

        agg = merged.groupby(['subject_id','hadm_id']).agg(
            max_mortality=('mortality_proxy','max'),
            mean_mortality=('mortality_proxy','mean'),
            count_comorbidities=('icd9_code','count')
        ).reset_index()

        agg = agg.fillna({
            'max_mortality': self.global_mean_,
            'mean_mortality': self.global_mean_,
            'count_comorbidities': 0
        })

        return df.merge(agg, on=['subject_id','hadm_id'], how='left').reset_index(drop = True)

class IndexResetter(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X.reset_index(drop=True)


In [1]:


test_col = "a"
monkeypatch.setattr(fe, "ICD9_DIAGNOSIS", test_col)

result = change_feature_names(df.copy())

assert result.loc[result['id'] == 1, "a"].iloc[0] == "323", "change_features: patient 1 wrong ICD9"
assert result.loc[result['id'] == 2, "a"].iloc[0] == "409", "change_features: patient 2 wrong ICD9"
assert result.loc[result['id'] == 3, "a"].iloc[0] == "TEE", "change_features: patient 3 wrong ICD9"
assert result.loc[result['id'] == 4, "a"].iloc[0].isna() == True, "change_features: patient 4 wrong ICD9"

IndentationError: unexpected indent (2444280031.py, line 8)

In [10]:
df = pd.DataFrame(
        {
            "FEATURE1": [1,2,3,4,5],
            "FEAT_2": [2,3,4,5,6],
            "ICD_FEATURE": ["44323", "54345", "grgrg", "432dw", None]

        }
    )

In [14]:
df.loc[df['FEATURE1'] == 1, 'ICD_FEATURE']

0    44323
Name: ICD_FEATURE, dtype: object

In [39]:
!source ../.venv/bin/activate

Python(72089) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [2]:
import sys
sys.executable


'/Users/gnlm/Desktop/ds_projects/Probability-of-Death-backup/.venv/bin/python'

In [27]:
import pandas as pd
import numpy as np
from probability_of_death.preprocessing.preprocessor import (
    drop_features,
    change_feature_names,
    change_comorbidities_icd9code)
from probability_of_death.feature_engineering.feature_engineering import (
    create_basic_features,
    encoder_demographics,
    encoder_icd9_codes,
    apply_icd9_mapping

)
from probability_of_death.config import DROP_FEATURES, ID_FEATURES, TRAIN_NUMERICAL_FEATURES, TRAIN_CATEGORICAL_FEATURES, TARGET

In [36]:
def random_dates(start, end, n):
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)

    return start + pd.to_timedelta(
        np.random.randint(0, (end - start).days, n), unit="D"
    )


def make_fake_patient_data(n=10):
    base_dates = pd.date_range("2020-01-01", periods=n, freq="D")
    rand_seconds = np.random.randint(0, 24*60*60, size=n)
    admit_times = base_dates + pd.to_timedelta(rand_seconds, unit="s")

    df = pd.DataFrame({
        'DOD': np.arange(n),
        'DISCHTIME': np.arange(n),
        'DEATHTIME': np.arange(n),
        'LOS': np.arange(n),
        "subject_id": np.arange(n),
        "hadm_id": np.arange(100, 100 + n),
        "ADMITTIME": admit_times,
        "RELIGION": ["NONE", "CATHOLIC", None, "JEWISH", "HINDU", "NOT SPECIFIED", "CATHOLIC", None, "UNOBTAINABLE", "HINDU"],
        "MARITAL_STATUS": ["UNKNOWN (DEFAULT)", None, "MARRIED", "DIVORCED", "WIDOWED", "LAF", None, "UNOBTAINABLE", "DIVORCED", "UNKNOWN (DEFAULT)"],
        "ETHNICITY": ["UNABLE TO OBTAIN", "WHITE", None, "BLACK", "HISPANIC", "UNABLE TO OBTAIN", "UNKNOWN/NOT SPECIFIED", None, "UNKNOWN/NOT SPECIFIED", "HISPANIC"],
        "INSURANCE": ["YES", "NO", "YES", "poe", "YES", "NO", "YES", "poe", "YES", "NO"],
        "GENDER": ["Male", None, "Female", "Declined", "None", "Male", None, "Female", None, "Declined)"],
        "ICD9_diagnosis": ["44323", "54345", None, "AB", "004", 2234, "54345", None, 4566, 23],
        "HOSPITAL_EXPIRE_FLAG": [1,0,0,0,1,1,0,0,1,0],
        "icustay_id": np.random.randint(10000,99999,size = n)
        # any required default columns
    })

    return df

def generate_fake_icd_codes(n):
    base_codes = ["44323", "54345", "25000", "41401", "V3000", "AB123"]
    return np.random.choice(base_codes, size=n)

def make_comorbidities_data(n_patients=10, codes_per_patient=3):
    # total rows = n_patients * codes_per_patient
    total = n_patients * codes_per_patient

    subject_ids = np.repeat(np.arange(n_patients), codes_per_patient)
    hadm_ids = np.repeat(np.arange(100, 100 + n_patients), codes_per_patient)

    df = pd.DataFrame({
        "SUBJECT_ID": subject_ids,
        "HADM_ID": hadm_ids,
        "SEQ_NUM": np.tile(np.arange(codes_per_patient), n_patients),
        "ICD9_CODE": generate_fake_icd_codes(total)
    })

    return df


In [43]:
test_df = make_fake_patient_data()
test_df["DOB"] = random_dates("1920-01-01", "2000-12-31", 10)
test_comorbidities_df = make_comorbidities_data()

In [44]:
def preprocess(df, comorbidities_df, icd9_mapping:None):
    df = create_basic_features(df)
    df = encoder_demographics(df)
    df = drop_features(df, DROP_FEATURES)
    df = change_feature_names(df)
    comorbidities_df = change_comorbidities_icd9code(comorbidities_df)
    df,mapping = encoder_icd9_codes(df, comorbidities_df)

    df = df.drop(ID_FEATURES, axis = 1)

    return df, mapping

In [45]:
test_df, test_mapping = preprocess(test_df, test_comorbidities_df, None)

/Users/gnlm/Desktop/ds_projects/Probability-of-Death-backup/src/probability_of_death/feature_engineering/feature_engineering.py:75: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df[MAX_MORTALITY] = train_df[MAX_MORTALITY].fillna(0)
/Users/gnlm/Desktop/ds_projects/Probability-of-Death-backup/src/probability_of_death/feature_engineering/feature_engineering.py:76: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pand

In [47]:
"HOSPITAL_EXPIRE_FLAG" in test_df.columns

True

In [31]:
train = pd.read_csv("/Users/gnlm/Desktop/ds_projects/Probability-of-Death-backup/data/mimic_train.csv")
train.head()

,HOSPITAL_EXPIRE_FLAG,subject_id,hadm_id,icustay_id,HeartRate_Min,HeartRate_Max,HeartRate_Mean,SysBP_Min,SysBP_Max,SysBP_Mean,...,Diff,ADMISSION_TYPE,INSURANCE,RELIGION,MARITAL_STATUS,ETHNICITY,DIAGNOSIS,ICD9_diagnosis,FIRST_CAREUNIT,LOS
0,0,55440,195768,228357,89.0,145.0,121.043478,74.0,127.0,106.586957,...,-61961.78470,EMERGENCY,Medicare,PROTESTANT QUAKER,SINGLE,WHITE,GASTROINTESTINAL BLEED,5789,MICU,4.5761
1,0,76908,126136,221004,63.0,110.0,79.117647,89.0,121.0,106.733333,...,-43146.18378,EMERGENCY,Private,UNOBTAINABLE,MARRIED,WHITE,ESOPHAGEAL FOOD IMPACTION,53013,MICU,0.7582
2,0,95798,136645,296315,81.0,98.0,91.689655,88.0,138.0,112.785714,...,-42009.96157,EMERGENCY,Medicare,PROTESTANT QUAKER,SEPARATED,BLACK/AFRICAN AMERICAN,UPPER GI BLEED,56983,MICU,3.7626
3,0,40708,102505,245557,76.0,128.0,98.857143,84.0,135.0,106.972973,...,-43585.37922,ELECTIVE,Medicare,NOT SPECIFIED,WIDOWED,WHITE,HIATAL HERNIA/SDA,5533,SICU,3.8734
4,0,28424,127337,225281,NaN,NaN,NaN,NaN,NaN,NaN,...,-50271.76602,EMERGENCY,Medicare,JEWISH,WIDOWED,WHITE,ABDOMINAL PAIN,56211,TSICU,5.8654
